## 1. Imports

In [28]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error,
)

# 2. Get the data

In [29]:
DF_PATH = Path("../data/processed/hour_feature_engineered.parquet")
df = pd.read_parquet(DF_PATH)

## 3. Split into training and test sets

In [30]:
train = df[(df["dteday"] >= pd.Timestamp(2011, 1, 8)) & (df["dteday"] <= pd.Timestamp(2012, 9, 30))]
valid = df[df["dteday"] > pd.Timestamp(2012, 9, 30)]

## 4. ARIMA model

In [31]:
arima_model = SARIMAX(
    train["cnt"],
    order=(3, 1, 1)
)
arima_results = arima_model.fit()

# predictions
arima_predictions = arima_results.forecast(steps=len(valid))

# evaluate
arima_metrics_df = pd.DataFrame(
    {
        "R² score": r2_score(valid["cnt"], arima_predictions),
        "MAE": mean_absolute_error(valid["cnt"], arima_predictions),
        "RMSE": root_mean_squared_error(valid["cnt"], arima_predictions),
    }, index=["ARIMA"]
)

# look at the metrics
arima_metrics_df.head()

,R² score,MAE,RMSE
ARIMA,-0.576609,178.846791,253.117565


## 5. SARIMA model

In [32]:
sarima_model = SARIMAX(
    train["cnt"],
    order=(3, 1, 1),
    seasonal_order=(1, 1, 1, 24)
)

sarima_results = sarima_model.fit()

# predictions
sarima_predictions = sarima_results.forecast(steps=len(valid))

# metrics
sarima_metrics_df = pd.DataFrame({
    "R² score": r2_score(valid["cnt"], sarima_predictions),
    "MAE": mean_absolute_error(valid["cnt"], sarima_predictions),
    "RMSE": root_mean_squared_error(valid["cnt"], sarima_predictions),
}, index=["SARIMA"])

# look at the results
sarima_metrics_df.head()

,R² score,MAE,RMSE
SARIMA,-1.344809,243.245048,308.683965


## 6. Merge the results

In [33]:
classical_forecasting_metrics_df = pd.concat([arima_metrics_df, sarima_metrics_df]).round(2)

# look at the metrics
classical_forecasting_metrics_df.head()

,R² score,MAE,RMSE
ARIMA,-0.58,178.85,253.12
SARIMA,-1.34,243.25,308.68


## 7. Predictions DataFrame

In [34]:
classical_models_predictions_df = pd.DataFrame({
    "actual": valid["cnt"],
    "ARIMA predictions": arima_predictions,
    "SARIMA predictions": sarima_predictions,
})

# look at the predictions DataFrame
classical_models_predictions_df

,actual,ARIMA predictions,SARIMA predictions
15043,45,54.332141,46.927981
15044,18,63.890584,49.462587
15045,12,69.043123,52.359609
15046,7,68.119624,37.249864
15047,10,66.035239,13.610513
...,...,...,...
17206,119,66.122213,72.410880
17207,89,66.122213,60.233430
17208,90,66.122213,83.671250
17209,61,66.122213,166.029366


## 8. Save the ARIMA and SARIMA metrics

In [35]:
CLASSICAL_FORECASTING_MODELS_PATH = Path("../data/processed/classical_forecasting_models_metrics.csv")
classical_forecasting_metrics_df.to_csv(CLASSICAL_FORECASTING_MODELS_PATH, index=True)

## 9. Save predictions

In [36]:
PREDICTIONS_PATH = Path("../data/processed/classical_forecasting_models_predictions.csv")
classical_models_predictions_df.to_csv(PREDICTIONS_PATH, index=False)